# GuardianLens — Live Demo

GuardianLens is an on-device AI child-safety monitor. It captures
the child's screen, runs **Gemma 4 (via Ollama)** locally to read the
chat content, and notifies the parent only when something is wrong.
Zero raw chat text leaves the device.

## What this notebook shows

1. A real Kaggle GPU runtime installs Ollama and pulls Gemma 4.
2. A staged frame stream is generated from six demo video clips
   spanning **TikTok DMs**, **Discord DMs**, and **Minecraft world chat** —
   a mix of safe peer chats (Emma + her friends), one Discord
   nitro-scam DM, in-game bullying, and a TikTok grooming attempt.
3. Frames are POSTed to the GuardianLens server at 1 Hz over the
   real `/api/frames` endpoint — same code path the deployed
   client uses.
4. The live dashboard reacts to each new frame: conversations
   appear in the activity panel, get re-classified as messages
   accumulate, and the bell + toasts fire when Gemma 4 detects
   grooming, bullying, or spam patterns.
5. After the stream ends, an observer walks through every flagged
   alert in turn so the AI's reasoning and parent-recommendations
   are easy to read.
6. The composed side-by-side recording (source on the left,
   dashboard reactions on the right) is saved to
   `/kaggle/working/guardianlens_demo_side_by_side.mp4` so you can
   download it from the notebook's **Output** tab.

## Try it with your own scenes

The frame stream comes from the Kaggle dataset
**`nataliajakowska/guardianlens-demo-screenshots`** — any `*.mp4` files
in that dataset are picked up automatically (sorted so files with
`safe` in the name play first). To test against your own footage:

1. Fork this notebook.
2. Replace the attached dataset with your own (any short MP4s of
   chat content — phone screen recordings work great).
3. Re-run. The pipeline will extract one frame per second from each
   clip in turn, send them to Gemma, and surface the same dashboard.

All steps are real — none of the dashboard activity is scripted on
the browser side. Code cells are hidden by default for readability;
click any one to expand its source.


In [ ]:
# ═══════════════════════════════════════════════
# STEP 1: Install Ollama and pull Gemma 4
# Takes ~10 minutes on first run
# ═══════════════════════════════════════════════
import subprocess, sys, os, time, shutil

# Verify the GPU allocation we got. Kaggle "GPU T4 x2" gives two
# T4 cards; if we got something else (P100, single T4) the inference
# story below changes. Print so the log is auditable.
print("=== GPU allocation ===")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout)

print("Installing zstd...")
subprocess.run("apt-get install -y zstd", shell=True, capture_output=True)

print("Installing Ollama...")
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True, text=True)

# Install script exits non-zero in containers (systemd fails) but binary is still placed.
OLLAMA = shutil.which("ollama") or "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA):
    print("Install stdout:", r.stdout[-400:])
    print("Install stderr:", r.stderr[-400:])
    raise RuntimeError("Ollama binary not found after install")
print(f"Ollama installed: {OLLAMA}")

os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

# Tell Ollama to spread layers across all visible GPUs. Without this,
# Ollama loads the full model on GPU 0 and the second T4 sits idle.
# Gemma 4 (~7 GB) fits on a single T4, but spreading reduces per-GPU
# memory pressure and lets Ollama serve concurrent requests faster.
ollama_env = {
    **os.environ,
    "OLLAMA_HOST": "0.0.0.0:11434",
    # No OLLAMA_SCHED_SPREAD: Gemma 4 (~7 GB) fits comfortably on a
    # single T4 (16 GB), and our pipeline issues calls sequentially.
    # Spreading across 2x T4 makes single-call inference SLOWER due
    # to cross-GPU coordination, with no benefit for our workload.
    # The second T4 sits idle — fine for a demo.
    "OLLAMA_NUM_PARALLEL": "2",
}

print("Starting Ollama server...")
ollama_proc = subprocess.Popen(
    [OLLAMA, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=ollama_env,
)

for i in range(30):
    time.sleep(2)
    probe = subprocess.run(
        ["curl", "-s", "--max-time", "2", "http://localhost:11434/api/version"],
        capture_output=True
    )
    if probe.returncode == 0:
        print(f"Ollama server ready (PID {ollama_proc.pid}) after {(i+1)*2}s")
        break
else:
    raise RuntimeError("Ollama server did not start within 60 s")

print("Pulling Gemma 4 latest (~17 GB, ~15 min)...")
r = subprocess.run([OLLAMA, "pull", "gemma4:latest"],
                   capture_output=True, text=True)
if r.returncode == 0:
    print("Model ready.")
else:
    print("Pull stderr:", r.stderr[-500:])
    raise RuntimeError("gemma4:latest pull failed")

# Pre-warm: load the model into VRAM with a dummy chat. Without
# this, the FIRST inference request from the dashboard subprocess
# pays the cold-start cost (~30-60 s on T4), eating the demo's
# recording budget. After this call the weights are resident.
print("\nWarming up Gemma 4 (loads weights into VRAM)...")
warm_start = time.time()
warm = subprocess.run(
    ["curl", "-s", "-X", "POST", "http://localhost:11434/api/chat",
     "-H", "Content-Type: application/json",
     "-d", '{"model":"gemma4:latest","messages":[{"role":"user","content":"hi"}],"stream":false}'],
    capture_output=True, text=True, timeout=300,
)
warm_secs = time.time() - warm_start
print(f"Model warm in {warm_secs:.1f}s" if warm.returncode == 0
      else f"Pre-warm failed (exit {warm.returncode}): {warm.stderr[:200]}")

# Post-warm nvidia-smi proves the model is resident on the GPU now,
# unlike a post-`pull` snapshot which would show 0 MiB used.
print("\n=== GPU memory after warm-up ===")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout)


In [ ]:
# ═══════════════════════════════════════════════
# STEP 2: Install GuardianLens + Playwright
# ═══════════════════════════════════════════════
import urllib.request, zipfile

print("Downloading GuardianLens...")
urllib.request.urlretrieve(
    "https://github.com/natalia-jaskowska/guardianlens/archive/refs/heads/main.zip",
    "/tmp/gl.zip"
)
with zipfile.ZipFile("/tmp/gl.zip") as z:
    z.extractall("/tmp/")

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-input", "/tmp/guardianlens-main"],
    capture_output=True, text=True
)
print("GuardianLens installed." if r.returncode == 0 else f"Error: {r.stderr[-300:]}")
sys.path.insert(0, "/tmp/guardianlens-main/src")

print("Installing Playwright...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"],
               capture_output=True)
r = subprocess.run(["playwright", "install", "chromium", "--with-deps"],
                   capture_output=True, text=True)
print("Chromium ready." if r.returncode == 0 else f"Chromium note: {r.stderr[-200:]}")


In [ ]:
# ═══════════════════════════════════════════════
# STEP 3: Imports and helpers
# ═══════════════════════════════════════════════
from pathlib import Path
from IPython.display import display, HTML
import json, base64

SCREENSHOTS = Path("/kaggle/input/guardianlens-demo-screenshots")

THREAT_COLORS = {
    "safe":     ("#22c55e", "#dcfce7"),
    "caution":  ("#f59e0b", "#fef9c3"),
    "warning":  ("#f97316", "#fff7ed"),
    "alert":    ("#ef4444", "#fee2e2"),
    "critical": ("#7c3aed", "#ede9fe"),
}
THREAT_EMOJI = {"safe": "✅", "caution": "⚠️", "warning": "🔶", "alert": "🚨", "critical": "🆘"}

def img_b64(path):
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode()

def render_card(analysis):
    level = analysis["threat_level"]
    color, bg = THREAT_COLORS[level]
    emoji = THREAT_EMOJI[level]
    indicators = analysis.get("indicators", [])
    ind_html = "".join(
        f'<span style="background:{color};color:white;padding:3px 10px;border-radius:12px;margin:3px;font-size:13px;display:inline-block">{i}</span>'
        for i in indicators
    ) or '<span style="color:#888">none</span>'
    return f"""
    <div style="max-width:900px;margin:24px 0;font-family:sans-serif;border:2px solid {color};border-radius:12px;overflow:hidden">
      <div style="background:{color};padding:10px 16px;display:flex;align-items:center;gap:10px">
        <span style="font-size:26px">{emoji}</span>
        <span style="font-size:20px;font-weight:bold;color:white">{level.upper()}</span>
        <span style="font-size:14px;color:rgba(255,255,255,0.85);margin-left:4px">confidence: {analysis['confidence']:.0f}%</span>
        <span style="margin-left:auto;font-size:13px;color:rgba(255,255,255,0.75)">{analysis.get('platform','')}</span>
      </div>
      <img src="{img_b64(SCREENSHOTS / analysis['screenshot'])}" style="width:100%;display:block"/>
      <div style="background:{bg};padding:16px">
        <div style="margin-bottom:10px"><b>Category:</b> {analysis.get('category','—')} &nbsp;·&nbsp;
        <b>Grooming stage:</b> {analysis.get('grooming_stage','none')}</div>
        <div style="margin-bottom:10px"><b>Indicators:</b><br><div style="margin-top:4px">{ind_html}</div></div>
        <div style="margin-bottom:10px"><b>Narrative:</b><br><span style="color:#333;line-height:1.6">{analysis.get('narrative','')}</span></div>
        <div style="background:rgba(0,0,0,0.05);border-radius:6px;padding:10px;font-size:13px;color:#555;line-height:1.6">
          <b>Gemma 4 reasoning:</b><br>{analysis.get('reasoning','')}
        </div>
      </div>
    </div>"""

print("Ready.")


## Architecture

```
Child's device (client)            Parent's machine (server)
───────────────────────            ─────────────────────────────────────
guardlens-client                   FastAPI + Ollama (Gemma 4)
  mss screen capture  ──PNG──▶      Step 1: extract_conversations  (vision)
  httpx sender                      Step 2: match & merge history  (deterministic)
                                    Step 3: update_conv_status     (reasoning)
                                    Step 4: generate_parent_alert  (if threat ≥ WARNING)
                                    SQLite ← all state persisted after each step
                                    SSE dashboard :7860
```

**Gemma 4 features used:**
- **Multimodal vision** — reads screenshots directly, no OCR, works on any platform
- **Native function calling** — typed, Pydantic-validated outputs, no regex
- **Thinking chain** — full reasoning shown to parents, explainable AI
- **Runs via Ollama** — zero cloud, child's data never leaves the home

In [ ]:
# ═══════════════════════════════════════════════
# STEP 4: Subsample frames + start the dashboard server
# ═══════════════════════════════════════════════
# Production-style flow: this kernel is the "client" (frame sender).
# The dashboard subprocess is the "server" — it owns the SQLite DB,
# the MonitorWorker, and the Gemma 4 pipeline. Frames travel between
# them over HTTP, exactly the same way the real guardlens-client
# pushes screenshots in the deployed product.
#
# Subsampling is deterministic: ffmpeg's `-vf fps=1/N` extracts one
# frame every N seconds of video time, regardless of the source fps.
# Output filenames are timestamp-zero-padded so they sort in the order
# they should be sent.

import os, sys, time, base64, subprocess, threading
from pathlib import Path
from IPython.display import display, HTML

DB_PATH = "/tmp/guardianlens_kaggle.db"
FRAMES  = Path("/tmp/frames"); FRAMES.mkdir(exist_ok=True)

# SUBMIT_INTERVAL_S = 1 means we send one frame per second to the server.
# The server keeps only the LATEST pending frame, so if processing is
# slow, intermediate frames are dropped — same semantics as a real-time
# monitoring agent that only cares about "what's on screen now".
SUBMIT_INTERVAL_S = 1     # POST one frame per second

# Discover all *.mp4 files in the dataset and order them so SAFE
# scenes come first, ALERT scenes second. We then CONCATENATE them
# into a single timeline and subsample at exactly 1 frame per second.
# The benefit: frame N on the wire corresponds to second N of the
# unified demo timeline — no per-video timestamp resets, no fractional
# offsets between scenes. Mirrors what a real screen-capture client
# (mss-based) sees: one continuous stream of screenshots.
VIDEOS = sorted(
    SCREENSHOTS.glob("*.mp4"),
    key=lambda p: (0 if "safe" in p.name else 1, p.name),
)
print(f"Found {len(VIDEOS)} videos (concat order):")
for v in VIDEOS:
    sz = v.stat().st_size // 1024
    print(f"  {v.name}  {sz} KB")

# Wipe previous frames.
for old in FRAMES.glob("*.png"):
    old.unlink()
combined_path = FRAMES / "combined.mp4"
if combined_path.exists():
    combined_path.unlink()

# Probe each video's dimensions for visibility (and so the log
# captures what the dataset actually contained).
def _probe_size(path):
    r = subprocess.run([
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height",
        "-of", "csv=p=0", str(path),
    ], capture_output=True, text=True, check=True)
    w, h = r.stdout.strip().split(",")
    return int(w), int(h)

print("  Source dimensions:")
for v in VIDEOS:
    w, h = _probe_size(v)
    print(f"    {v.name}: {w}x{h}")

# Normalize every input to the source-column size (600 x 1080) used by
# the side-by-side composite below. Doing this in concat (rather than
# on max-of-inputs and rescaling later) avoids a double-scale and
# keeps each scene as large as possible inside the column:
#   * TikTok 460x900 (portrait)  -> 552x1080, 24 px black on each side
#   * Minecraft 1920x1080 (16:9) -> 600x337, 371 px black top+bottom
SOURCE_COL_W, SOURCE_COL_H = 600, 1080
print(f"  Normalizing all to {SOURCE_COL_W}x{SOURCE_COL_H} (source-column size)")
parts = [
    f"[{i}:v]scale={SOURCE_COL_W}:{SOURCE_COL_H}:force_original_aspect_ratio=decrease,"
    f"pad={SOURCE_COL_W}:{SOURCE_COL_H}:(ow-iw)/2:(oh-ih)/2:black,"
    f"setsar=1,fps=25[v{i}]"
    for i in range(len(VIDEOS))
]
concat_inputs = "".join(f"[v{i}]" for i in range(len(VIDEOS)))
parts.append(f"{concat_inputs}concat=n={len(VIDEOS)}:v=1:a=0[out]")
input_args = []
for v in VIDEOS:
    input_args.extend(["-i", str(v)])
subprocess.run([
    "ffmpeg", "-y",
    *input_args,
    "-filter_complex", ";".join(parts),
    "-map", "[out]",
    "-c:v", "libx264", "-preset", "fast", "-crf", "22",
    "-pix_fmt", "yuv420p",
    str(combined_path),
], check=True, capture_output=True)

# ── Analysis frames: per-video NATIVE resolution (no concat scale) ──
# The server-side pipeline needs the sharpest possible input. We do
# NOT reuse the 600x1080 display concat above for analysis — it
# letterboxes Minecraft 1920x1080 to 600x337, which crushes chat text
# from ~16 px to ~5 px (unreadable by Gemma vision). Instead, extract
# one frame per second from EACH source at its native resolution.
#
# Submission schedule preserves true cumulative wall-clock: if video 1
# is 20.5 s long, video 2's first frame is submitted at t=20.5 s, then
# 21.5, 22.5, etc. This keeps on-screen scene boundaries aligned with
# the side-by-side display video.
def _probe_duration(p):
    r = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=nw=1:nk=1", str(p),
    ], capture_output=True, text=True, check=True)
    return float(r.stdout.strip())

FRAME_PLAN = []          # [(submit_time_s, path), ...]
cumulative = 0.0
for vi, v in enumerate(VIDEOS):
    D = _probe_duration(v)
    outdir = FRAMES / f"v{vi}_{v.stem}"
    outdir.mkdir(parents=True, exist_ok=True)
    for old in outdir.glob("frame_*.png"):
        old.unlink()
    subprocess.run([
        "ffmpeg", "-y", "-i", str(v),
        "-vf", "fps=1", "-q:v", "2",
        str(outdir / "frame_%03d.png"),
    ], check=True, capture_output=True)
    paths = sorted(outdir.glob("frame_*.png"))
    for idx, p in enumerate(paths):
        FRAME_PLAN.append((cumulative + idx * SUBMIT_INTERVAL_S, p))
    print(f"  {v.name}: {D:.2f}s native, {len(paths)} frames, starts at t={cumulative:.2f}s")
    cumulative += D

TOTAL_TIMELINE_S = cumulative
print(f"\nTotal analysis plan: {len(FRAME_PLAN)} frames, {TOTAL_TIMELINE_S:.2f}s timeline")

# ── Start the dashboard server (receive mode) ─────────────────────────
dash_env = {**os.environ,
            "GUARDLENS_DATABASE__PATH": DB_PATH,
            "GUARDLENS_OLLAMA__HOST": "http://localhost:11434"}

# Capture dashboard logs to a file so we can grep timestamps for
# frame receive / process-start / process-end events. Switch log level
# to INFO to enable the relevant logger.info calls in the worker.
DASH_LOG = Path("/tmp/dashboard.log")
DASH_LOG.write_text("")  # truncate from any prior run
_dash_log_fp = DASH_LOG.open("ab")
dash_proc = subprocess.Popen(
    [sys.executable, "/tmp/guardianlens-main/run.py",
     "--no-capture", "--model", "gemma4:latest", "--dashboard-port", "7860", "--log-level", "INFO"],
    env=dash_env, stdout=_dash_log_fp, stderr=subprocess.STDOUT,
)
for i in range(25):
    time.sleep(2)
    if subprocess.run(["curl", "-s", "--max-time", "2", "http://localhost:7860/"],
                      capture_output=True).returncode == 0:
        print(f"\nDashboard ready after {(i+1)*2}s")
        break
else:
    raise RuntimeError("Dashboard did not start")


In [ ]:
# ═══════════════════════════════════════════════
# STEP 5: Live demo — sender + Playwright observer
# ═══════════════════════════════════════════════
# Two threads in this kernel:
#   - Sender: POSTs each subsampled frame to /api/frames at
#     SUBMIT_INTERVAL_S cadence. The server's MonitorWorker has a
#     latest-only slot — if the pipeline is mid-inference when a
#     newer frame lands, the older pending frame is dropped (matches
#     real-time monitoring semantics).
#   - Playwright observer (separate subprocess): records the dashboard
#     headlessly while the sender runs.
#
# Deterministic by design: ffmpeg subsampling + 1-Hz HTTP posts.
# Inference timing is the only source of variance.

import httpx
from urllib.parse import urljoin

SHOTS     = Path("/tmp/app_shots");  SHOTS.mkdir(exist_ok=True)
VIDEO_DIR = Path("/tmp/app_video");  VIDEO_DIR.mkdir(exist_ok=True)
SERVER    = "http://localhost:7860"

# Recording duration is deterministic: sender pushes the source at
# 1 frame/s for SOURCE_DURATION_S seconds (= source video length),
# then we record TAIL_DURATION_S more seconds for late inferences to
# finish + a closing-beat bell-menu open. The composed video ends
# up exactly SOURCE_DURATION_S + TAIL_DURATION_S long. Source side
# of the composite plays through then goes black for the tail.
SOURCE_DURATION_S = TOTAL_TIMELINE_S                       # true sum of video durations
TAIL_DURATION_S   = 75    # ~12 s per flagged-card walkthrough * up to 5 cards + bell beat
RECORD_BUDGET_S   = SOURCE_DURATION_S + TAIL_DURATION_S
print(f"Source: {SOURCE_DURATION_S}s + tail: {TAIL_DURATION_S}s = recording {RECORD_BUDGET_S}s")

# ── Playwright observer (records the dashboard) ──────────────────────
observer_script = r"""
import asyncio, sys
from playwright.async_api import async_playwright
from pathlib import Path

SHOTS = Path("/tmp/app_shots")
RECORD_S = float(sys.argv[1])
SOURCE_DURATION_S = float(sys.argv[2])

# Mirror every OBSERVER print to a log file the kernel can read at
# the end. Subprocess stdout is consumed silently; without this we
# can't tell why the post-stream walkthrough did or didn't fire.
_OBS_LOG = open("/tmp/observer.log", "w")
import builtins as _b
_orig_print = _b.print
def _logged_print(*args, **kwargs):
    _orig_print(*args, **kwargs)
    try:
        msg = " ".join(str(a) for a in args)
        _OBS_LOG.write(msg + "\n"); _OBS_LOG.flush()
    except Exception:
        pass
_b.print = _logged_print

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context(
            viewport={"width": 1400, "height": 900},
            record_video_dir="/tmp/app_video/",
            record_video_size={"width": 1400, "height": 900},
        )
        # Recording starts the moment new_context() is called above.
        # Capture that timestamp so the kernel can trim the lead-in
        # when composing the source video alongside the dashboard.
        import time as _t
        record_started = _t.monotonic()
        page = await context.new_page()
        await page.goto("http://localhost:7860", wait_until="domcontentloaded", timeout=20000)
        await page.wait_for_function(
            "() => typeof ui !== 'undefined' && ui.knownAlerts instanceof Set",
            timeout=15000,
        )
        await page.screenshot(path=str(SHOTS / "00_clean.png"))
        lead_in = _t.monotonic() - record_started
        print(f"READY {lead_in:.3f}", flush=True)

        # Wait for an alert to fire on the dashboard. Poll /api/state
        # so we know the moment the parent-facing UI flips into alert
        # tone — that's when we drill in. Cap the wait so we still
        # leave time for the read-through choreography afterward.
        WAIT_BUDGET = max(5.0, RECORD_S - 22.0)
        alert_t = None
        poll_started = _t.monotonic()
        while _t.monotonic() - poll_started < WAIT_BUDGET:
            try:
                snap = await page.evaluate(
                    "fetch('/api/state').then(r => r.json())"
                )
            except Exception:
                snap = {}
            if snap.get("is_alert"):
                alert_t = _t.monotonic()
                print(f"OBSERVER alert detected after {alert_t - poll_started:.1f}s", flush=True)
                break
            await asyncio.sleep(0.5)

        if alert_t is not None:
            # First touch — quick drill-in so the camera shows the
            # parent reacting to the alert toast in real time. Stay
            # brief; the read-through walkthrough happens after the
            # source video finishes.
            await asyncio.sleep(1.5)
            try:
                # Same JS-click workaround — toast overlay was
                # blocking the actionability check during the live
                # alert phase too.
                clicked = await page.evaluate(
                    """() => {
                        const sel = '.gl-card.conv.alert, .gl-card.conv.warning, .gl-card.conv';
                        const c = document.querySelectorAll(sel)[0];
                        if (!c) return false;
                        c.click();
                        return true;
                    }"""
                )
                if not clicked:
                    print("OBSERVER no card to drill into yet", flush=True)
                await asyncio.sleep(2.0)
                await page.evaluate("window.scrollTo({top: 0, behavior: 'smooth'})")
                await asyncio.sleep(1.0)
                # Pop back to the activity overview so the source video
                # keeps drawing attention while it plays out.
                back = page.locator("button.gl-back").first
                if await back.count():
                    await back.click()
                    await asyncio.sleep(0.5)
            except Exception as e:
                print(f"OBSERVER first-alert drill failed: {e}", flush=True)
        else:
            print("OBSERVER no alert observed within wait budget", flush=True)

        # ── Walk through EVERY flagged conversation at the end ──
        # After the source plays out, enumerate flagged cards in the
        # activity panel and click each one in turn so the camera
        # captures the full AI reasoning + recommendations for each.
        wait_until_source_ends = SOURCE_DURATION_S - (_t.monotonic() - record_started) + 1.0
        if wait_until_source_ends > 0:
            print(f"OBSERVER waiting {wait_until_source_ends:.1f}s for source end", flush=True)
            await asyncio.sleep(wait_until_source_ends)

        FLAGGED_SEL = (
            ".gl-card.conv.caution, .gl-card.conv.warning, "
            ".gl-card.conv.alert, .gl-card.conv.critical"
        )

        # First make sure we're on the overview, otherwise the cards
        # are in the activity panel BUT the right pane is showing a
        # detail — clicking a card from this state still works, but
        # the back button gets us to a clean baseline.
        try:
            await page.evaluate(
                "() => document.querySelector('button.gl-back')?.click()"
            )
            await asyncio.sleep(0.5)
        except Exception:
            pass

        # Snapshot the count up front. The activity panel is on the
        # left and renders persistently, so cards are reachable
        # regardless of right-pane state.
        try:
            n_flagged = await page.locator(FLAGGED_SEL).count()
        except Exception as e:
            print(f"OBSERVER count failed: {e}", flush=True)
            n_flagged = 0
        # Fallback: if no flagged cards, walk through ALL cards anyway
        # so the demo still shows a card-by-card review at the end.
        if n_flagged == 0:
            try:
                n_flagged = await page.locator(".gl-card.conv").count()
                print(f"OBSERVER no flagged cards — walking through {n_flagged} total cards", flush=True)
                CARD_SEL = ".gl-card.conv"
            except Exception:
                CARD_SEL = FLAGGED_SEL
        else:
            print(f"OBSERVER flagged cards to walk: {n_flagged}", flush=True)
            CARD_SEL = FLAGGED_SEL

        per_card_budget = 12.0
        for idx in range(n_flagged):
            remaining = RECORD_S - (_t.monotonic() - record_started)
            if remaining < per_card_budget + 6:
                print(f"OBSERVER stop — {remaining:.1f}s left", flush=True)
                break
            try:
                # Back to overview via JS (no-op when already there).
                await page.evaluate(
                    "() => document.querySelector('button.gl-back')?.click()"
                )
                await asyncio.sleep(0.4)
                cards = page.locator(CARD_SEL)
                cnt = await cards.count()
                if idx >= cnt:
                    print(f"OBSERVER ran out — wanted #{idx} of {cnt}", flush=True)
                    break
                target = cards.nth(idx)
                await target.scroll_into_view_if_needed(timeout=3000)
                print(f"OBSERVER clicking card #{idx}", flush=True)
                # JS-fire the click. Playwright's actionability check
                # was timing out 30 s waiting for a hovering toast /
                # alert-pulse animation to clear. The card itself is
                # always visible and clickable from JS regardless.
                await page.evaluate(
                    "(i) => document.querySelectorAll(arguments[0])[i].click()",
                    [CARD_SEL, idx],
                )
                await asyncio.sleep(2.0)
                await page.evaluate(
                    "document.querySelector('#stateConversation')?.scrollIntoView({behavior:'smooth', block:'start'})"
                )
                await asyncio.sleep(3.0)
                await page.evaluate("window.scrollBy({top: 380, behavior: 'smooth'})")
                await asyncio.sleep(3.5)
                await page.evaluate("window.scrollBy({top: 380, behavior: 'smooth'})")
                await asyncio.sleep(3.5)
                await page.evaluate("window.scrollTo({top: 0, behavior: 'smooth'})")
                await asyncio.sleep(0.8)
                print(f"OBSERVER card #{idx} done", flush=True)
            except Exception as e:
                print(f"OBSERVER card #{idx} failed: {e}", flush=True)

        # Closing beat: open the bell menu — final frame is the
        # full alerts inbox so the viewer sees everything Gemma
        # surfaced, in one shot.
        try:
            back = page.locator("button.gl-back").first
            if await back.count():
                await back.click()
                await asyncio.sleep(0.3)
        except Exception:
            pass
        remaining = RECORD_S - (_t.monotonic() - record_started)
        if remaining > 6:
            await asyncio.sleep(remaining - 6)
        try:
            await page.locator("#bellWrap").click()
        except Exception as e:
            print(f"OBSERVER bell-click failed: {e}", flush=True)
        await asyncio.sleep(4.0)
        await page.screenshot(path=str(SHOTS / "99_bell_menu.png"))
        await asyncio.sleep(1.0)

        await context.close()
        await browser.close()

asyncio.run(main())
"""
Path("/tmp/take_shots.py").write_text(observer_script)

pw_proc = subprocess.Popen(
    [sys.executable, "/tmp/take_shots.py", str(RECORD_BUDGET_S), str(SOURCE_DURATION_S)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

# Wait for observer to be READY before we start sending.
import re as _re
ready = False
PLAYWRIGHT_LEAD_IN = 0.0
start = time.monotonic()
while time.monotonic() - start < 30:
    line = pw_proc.stdout.readline()
    if not line:
        break
    m = _re.match(r"READY ([\d.]+)", line.strip())
    if m:
        PLAYWRIGHT_LEAD_IN = float(m.group(1))
        ready = True
        break
if not ready:
    err = pw_proc.stderr.read() if pw_proc.stderr else ""
    print("Playwright failed to reach READY state")
    print(err[-500:])
    pw_proc.kill()
    raise RuntimeError("Playwright did not signal ready")

print("Observer ready — beginning frame stream.")

# ── Sender: post on the wall-clock second-mark ───────────────────────
# Frame i is sent at sender_t0 + i*SUBMIT_INTERVAL_S, NOT at
# (previous-send + SUBMIT_INTERVAL_S). HTTP/network latency therefore
# does not accumulate — frame N goes out exactly N seconds after the
# first one regardless of what happened in between. This matches the
# determinism property we want: frame index = wall-clock second on the
# unified video timeline.
sender_t0 = time.monotonic()
def send_frames():
    with httpx.Client(timeout=10.0) as http:
        for target_t, frame_path in FRAME_PLAN:
            now_rel = time.monotonic() - sender_t0
            if now_rel < target_t:
                time.sleep(target_t - now_rel)
            try:
                with open(frame_path, "rb") as fh:
                    r = http.post(
                        urljoin(SERVER, "/api/frames"),
                        files={"file": (frame_path.name, fh, "image/png")},
                    )
                ok = r.status_code in (200, 202)
                t_elapsed = time.monotonic() - sender_t0
                print(f"  [t={t_elapsed:5.2f}s] POST {frame_path.name}: HTTP {r.status_code}{'' if ok else ' !!'}", flush=True)
            except Exception as exc:
                print(f"  POST {frame_path.name}: ERROR {exc}", flush=True)

sender = threading.Thread(target=send_frames, daemon=True)
sender.start()

# ── Wait for both ────────────────────────────────────────────────────
stdout, stderr = pw_proc.communicate(timeout=RECORD_BUDGET_S + 30)
sender.join(timeout=10)
if pw_proc.returncode != 0:
    print("Playwright stderr:", (stderr or "")[-500:])

# ── Encode the recording for inline playback ─────────────────────────
webm_files = sorted(VIDEO_DIR.glob("*.webm"))
if webm_files:
    dashboard_webm = webm_files[0]
    side_by_side_path = Path("/tmp/app_demo_side_by_side.mp4")

    # Compose source + dashboard side-by-side. Layout: 2400x1080.
    # Left  600 px  (25%) — the source TikTok video at native portrait
    #                       aspect, padded into a 600x1080 column.
    # Right 1800 px (75%) — the dashboard recording, scaled to fit
    #                       1080 high, padded width to 1800.
    # Sync: skip PLAYWRIGHT_LEAD_IN seconds of dashboard so its frame 1
    # corresponds in time to source frame 1 (sender's first POST).
    # Length: source freezes on its last frame after it ends; whole
    # output is cut at the dashboard's natural length.
    print(f"Composing side-by-side (dashboard skip={PLAYWRIGHT_LEAD_IN:.2f}s)...")
    encode = subprocess.run([
        "ffmpeg", "-y",
        "-i", str(combined_path),
        "-ss", f"{PLAYWRIGHT_LEAD_IN:.3f}", "-i", str(dashboard_webm),
        "-filter_complex",
        (
            # Left: source video is already 600x1080 (normalized at
            # concat time). Just append TAIL_DURATION_S seconds of
            # BLACK (tpad stop_mode=add) and overlay an "END OF STREAM"
            # caption gated to the tail only.
            f"[0:v]tpad=stop_mode=add:stop_duration={TAIL_DURATION_S},"
            f"drawtext=text='END OF STREAM':"
            f"fontcolor=white:fontsize=42:"
            f"x=(w-text_w)/2:y=(h-text_h)/2:"
            f"enable='gte(t,{SOURCE_DURATION_S})'[left];"
            # Right: dashboard scaled to 1080 high then padded to the
            # 1800-wide column.
            f"[1:v]scale=-2:1080,"
            f"pad=1800:1080:(1800-iw)/2:0:black[right];"
            # hstack=shortest=1 ends with the shorter input. Both
            # inputs run for ~SOURCE_DURATION_S+TAIL_DURATION_S, so
            # output is exactly that.
            f"[left][right]hstack=inputs=2:shortest=1,setpts=PTS-STARTPTS[out]"
        ),
        "-map", "[out]",
        "-c:v", "libx264", "-crf", "22", "-preset", "fast",
        "-movflags", "+faststart", "-pix_fmt", "yuv420p",
        "-shortest",
        str(side_by_side_path),
    ], capture_output=True, text=True)

    if side_by_side_path.exists():
        size_kb = side_by_side_path.stat().st_size // 1024
        # Persist to /kaggle/working so the recorded MP4 is downloadable
        # from the kernel's Output tab (anything in /kaggle/working is
        # surfaced as a kernel output artifact after the run finishes).
        # Fall back to /tmp when not running inside a Kaggle kernel.
        working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/tmp")
        download_path = working / "guardianlens_demo_side_by_side.mp4"
        try:
            import shutil as _sh
            _sh.copy2(side_by_side_path, download_path)
            print(f"\n  📺  Side-by-side video saved: {download_path}  ({size_kb} KB)")
            print(f"      Open the kernel's Output tab and download the MP4 above —")
            print(f"      it plays back at full HD which makes the dashboard text easier")
            print(f"      to read than the inline preview below.\n")
        except Exception as _exc:
            print(f"Side-by-side video: {size_kb} KB (copy to working failed: {_exc})")
        mp4_b64 = base64.b64encode(side_by_side_path.read_bytes()).decode()
        display(HTML(f"""
<div style="max-width:1200px;margin:16px 0;font-family:sans-serif">
  <div style="background:#0f172a;color:#f1f5f9;padding:12px 18px;
              border-radius:8px 8px 0 0;font-size:15px;font-weight:bold">
    GuardianLens Live — source (left) + dashboard reaction (right)
    <span style="font-weight:normal;font-size:12px;color:#94a3b8;margin-left:10px">
      2× T4 · {len(FRAME_PLAN)} frames @ 1 Hz · sync offset {PLAYWRIGHT_LEAD_IN:.1f}s
    </span>
  </div>
  <video controls autoplay loop muted
         style="width:100%;display:block;border-radius:0 0 8px 8px;background:#000">
    <source src="data:video/mp4;base64,{mp4_b64}" type="video/mp4"/>
  </video>
</div>"""))
    else:
        print("Side-by-side encode failed:")
        print(encode.stderr[-600:])
        # Fallback: show dashboard-only encode.
        mp4_path = Path("/tmp/app_demo.mp4")
        subprocess.run([
            "ffmpeg", "-y", "-i", str(dashboard_webm),
            "-vf", "scale=1280:-2",
            "-c:v", "libx264", "-crf", "22", "-preset", "fast",
            "-movflags", "+faststart", "-pix_fmt", "yuv420p",
            str(mp4_path),
        ], capture_output=True, text=True)
        if mp4_path.exists():
            mp4_b64 = base64.b64encode(mp4_path.read_bytes()).decode()
            display(HTML(f'<video controls autoplay loop muted style="width:100%"><source src="data:video/mp4;base64,{mp4_b64}" type="video/mp4"/></video>'))
else:
    print("No WebM recorded — showing screenshots")
    for fname in sorted(SHOTS.glob("*.png")):
        b64 = "data:image/png;base64," + base64.b64encode(fname.read_bytes()).decode()
        display(HTML(f'<img src="{b64}" style="width:100%;margin:4px 0;border-radius:6px"/>'))

# ── Dashboard log highlights — frame-level timing ────────────────────
# Receives, drops, process-starts, and process-ends from the dashboard
# subprocess. Use these to reason about per-frame latency.
print()
print("=== Dashboard server log (frame-level events) ===")
log_path = Path("/tmp/dashboard.log")
if log_path.exists():
    keep = ("Received frame", "Dropping pending", "Processing screenshot",
            "Pipeline done", "Pipeline failed",
            "LLM ", "Conversation unchanged",
            "Matched fragment", "Creating new conversation")
    for line in log_path.read_text().splitlines():
        if any(k in line for k in keep):
            print("  " + line)
else:
    print("  (no log captured)")

# Observer's own decisions (alert detection, walkthrough) — captured
# from /tmp/observer.log so we can see whether the post-stream
# card-by-card walk through actually fired.
print()
print("=== Playwright observer log ===")
obs_log = Path("/tmp/observer.log")
if obs_log.exists():
    for line in obs_log.read_text().splitlines():
        print("  " + line)
else:
    print("  (no observer log)")
